In [7]:
import pandas as pd
import numpy as np
import os
processed_path=r"data"

parquet_files=[]
for root ,dir,files in os.walk(processed_path):
    for file in files:
        if file.endswith(".parquet"):
            parquet_files.append(os.path.join(root,file))
# COLUMN ANALYSIS
all_columns = {}
for file in parquet_files:
    df=pd.read_parquet(file)

    print("\n" + "="*80)
    print("File",os.path.basename(file))
    print("="*80)

    print("\nTotal Columns:",len(df.columns))

    for i, col in enumerate(df.columns,1):
        print(f"{i:3}.{col}")
    all_columns[os.path.basename(file)] = list(df.columns)

    #Common Column
if all_columns:
    comman_column=set.intersection(
        *[set(col) for col in all_columns.values()]
    )
    print("\n"+ "="*80)
    print("Common Columns across all Files")
    print("="*80)

    print(f"Total Common Columns : {len(comman_column)}") 

    for col in sorted(comman_column):
        print("")
        print(col)    

# POSSIBLE LEAKAGE / IDENTIFIER COLUMNS

possible_leakage = [
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Src IP",
    "Dst IP",
    "Source Port",
    "Destination Port",
    "Src Port",
    "Dst Port"
]

print("\n "+ "="*80)
print("POSSIBLE LEAKAGE / IDENTIFIER COLUMNS")
print("="*80)

found_leakage=[]

if all_columns:
    for col in sorted(all_columns):
        if col.lower() in [x.lower() for x in possible_leakage]:
            found_leakage.append(col)
        elif "ip" in col.lower():
            found_leakage.append(col)
        elif "flow id" in col.lower():
            found_leakage.append(col)
    if found_leakage:
     for col in found_leakage:
            print(col)
     else:
        print("No obvious identifier columns found.")


# TIMESTAMP COLUMN
print("\n" + "=" * 80)
print("TIMESTAMP COLUMNS")
print("=" * 80)

timestamp_columns = []

for col in sorted(comman_column):

    if any(word in col.lower() for word in ["timestamp", "time", "date"]):
        timestamp_columns.append(col)

if timestamp_columns:
    for col in timestamp_columns:
        print(col)
else:
    print("No obvious timestamp column found.")

# CONSTANT COLUMNS
print("\n" + "=" * 80)
print("CONSTANT COLUMNS")
print("=" * 80)

constant_columns = set()

for file in parquet_files:

    df = pd.read_parquet(file)

    for col in df.columns:

        if df[col].nunique(dropna=False) <= 1:
            constant_columns.add(col)

if constant_columns:

    for col in sorted(constant_columns):
        print(col)

else:
    print("No constant columns found.")


print("\n" + "=" * 80)
print("FEATURE ANALYSIS COMPLETED")
print("=" * 80)


File Benign-Monday-no-metadata.parquet

Total Columns: 78
  1.Protocol
  2.Flow Duration
  3.Total Fwd Packets
  4.Total Backward Packets
  5.Fwd Packets Length Total
  6.Bwd Packets Length Total
  7.Fwd Packet Length Max
  8.Fwd Packet Length Min
  9.Fwd Packet Length Mean
 10.Fwd Packet Length Std
 11.Bwd Packet Length Max
 12.Bwd Packet Length Min
 13.Bwd Packet Length Mean
 14.Bwd Packet Length Std
 15.Flow Bytes/s
 16.Flow Packets/s
 17.Flow IAT Mean
 18.Flow IAT Std
 19.Flow IAT Max
 20.Flow IAT Min
 21.Fwd IAT Total
 22.Fwd IAT Mean
 23.Fwd IAT Std
 24.Fwd IAT Max
 25.Fwd IAT Min
 26.Bwd IAT Total
 27.Bwd IAT Mean
 28.Bwd IAT Std
 29.Bwd IAT Max
 30.Bwd IAT Min
 31.Fwd PSH Flags
 32.Bwd PSH Flags
 33.Fwd URG Flags
 34.Bwd URG Flags
 35.Fwd Header Length
 36.Bwd Header Length
 37.Fwd Packets/s
 38.Bwd Packets/s
 39.Packet Length Min
 40.Packet Length Max
 41.Packet Length Mean
 42.Packet Length Std
 43.Packet Length Variance
 44.FIN Flag Count
 45.SYN Flag Count
 46.RST Flag Cou

In [18]:
processed_path=r"data"
featured_path=r"featured selected"

os.makedirs(featured_path, exist_ok=True)


# Constant columns identified from previous analysis
constant_columns = [
    "Bwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "CWE Flag Count",
    "Fwd Avg Bulk Rate",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd URG Flags"
]

for root ,dir,files in os.walk(processed_path):
    for file in files:
        if file.endswith(".parquet"):
            file_path = os.path.join(root, file)

            df = pd.read_parquet(file_path)


    # Find constant columns present in this file
            col_to_remove = [
        col for col in constant_columns
        if col in df.columns
    ]


        df=df.drop(columns=col_to_remove)


#Save Featured Selected Dataset
        output_file=os.path.join(
    featured_path,file)

        df.to_parquet(output_file,index=False)
        print("=" * 70)
        print("File:", os.path.basename(file))
        print("Removed:", len(col_to_remove))
        print("Remaining columns:", len(df.columns))
        print("Saved:", output_file)

print("\nFeature selection completed.")

File: Benign-Monday-no-metadata.parquet
Removed: 10
Remaining columns: 68
Saved: featured selected\Benign-Monday-no-metadata.parquet
File: Botnet-Friday-no-metadata.parquet
Removed: 10
Remaining columns: 68
Saved: featured selected\Botnet-Friday-no-metadata.parquet
File: Bruteforce-Tuesday-no-metadata.parquet
Removed: 10
Remaining columns: 68
Saved: featured selected\Bruteforce-Tuesday-no-metadata.parquet
File: DDoS-Friday-no-metadata.parquet
Removed: 10
Remaining columns: 68
Saved: featured selected\DDoS-Friday-no-metadata.parquet
File: DoS-Wednesday-no-metadata.parquet
Removed: 10
Remaining columns: 68
Saved: featured selected\DoS-Wednesday-no-metadata.parquet
File: Infiltration-Thursday-no-metadata.parquet
Removed: 10
Remaining columns: 68
Saved: featured selected\Infiltration-Thursday-no-metadata.parquet
File: Portscan-Friday-no-metadata.parquet
Removed: 10
Remaining columns: 68
Saved: featured selected\Portscan-Friday-no-metadata.parquet
File: WebAttacks-Thursday-no-metadata.parqu